# Correlated Predictors: Event-Window Fundamental News Test

Implements the event-window design: analyst EPS revision news, lagged-price scaling, 5-trading-day predictor exposures, firm-predictor fundamental relevance, and LASSO selection-rate tests.

In [1]:

from pathlib import Path
import json
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import h5py
import statsmodels.api as sm

ROOT = Path.cwd(); DATA = ROOT / 'Data'; CLEAN = DATA / 'clean_data'
RESULTS = ROOT / 'Results' / 'Estimation' / 'Cross_Sectional'
OUT = RESULTS / 'correlated_predictors'; OUT.mkdir(parents=True, exist_ok=True)
IBES_PATH = CLEAN / 'ibes_detail_eps_forecasts_2010_2017.csv'
FEATURE_PATH = CLEAN / 'final_macro_topic_features.csv'
BETAS_PATH = RESULTS / 'betas.h5'
LOGIN_PATH = ROOT / 'wrds_login.txt'
PRICE_CACHE = CLEAN / 'crsp_daily_prices_2010_2017_sample.csv'
FUND_NEWS_CACHE = CLEAN / 'ibes_quarterly_fundamental_news_price_scaled.csv'
FUNDREL_CACHE = CLEAN / 'firm_predictor_fundrel_event_window_h5.csv'
FINAL_PANEL_PATH = CLEAN / 'correlated_predictors_firm_predictor_panel.csv'
SUMMARY_PATH = OUT / 'correlated_predictors_summary.json'
START, END = '2010-01-01', '2017-12-31'
WINDOW = 5; MIN_EVENTS_PER_FIRM = 20
print(ROOT)


C:\Users\jonat\Lasso_paper


## 1. Load LASSO Selection Rates

In [2]:

with h5py.File(BETAS_PATH, 'r') as f:
    stocks = np.array([int(x.decode() if isinstance(x, bytes) else x) for x in f['stocks'][:]])
    topics = np.array([x.decode() if isinstance(x, bytes) else str(x) for x in f['topics'][:]])
    dates = pd.to_datetime([x.decode() if isinstance(x, bytes) else x for x in f['dates'][:]])
    betas = f['betas'][:]
valid = np.isfinite(betas); selected = valid & (betas != 0)
selection_rate_mat = np.divide(selected.sum(axis=2), valid.sum(axis=2), out=np.full(selected.shape[:2], np.nan), where=valid.sum(axis=2)>0)
selection_df = pd.DataFrame(selection_rate_mat, index=stocks, columns=topics).stack(dropna=False).reset_index()
selection_df.columns = ['permno', 'predictor', 'SelectionRate']
selection_df = selection_df.dropna(subset=['SelectionRate'])
print(f'Beta tensor: stocks={len(stocks):,}, predictors={len(topics):,}, dates={len(dates):,}')
print(f'Selection panel rows: {len(selection_df):,}')
print(f'Mean selection rate: {selection_df.SelectionRate.mean()*100:.3f}%')


Beta tensor: stocks=1,296, predictors=190, dates=1,657
Selection panel rows: 246,240
Mean selection rate: 1.950%


## 2. Fetch/Load CRSP Prices for Lagged Scaling

In [3]:

def fetch_crsp_prices_if_needed():
    if PRICE_CACHE.exists():
        print(f'Using cached prices: {PRICE_CACHE}')
        return pd.read_csv(PRICE_CACHE, parse_dates=['date'])
    import wrds
    lines = LOGIN_PATH.read_text().splitlines(); user, pwd = lines[0].strip(), lines[1].strip()
    permno_list = ','.join(map(str, stocks.tolist()))
    sql = f"""
        select permno, date, abs(prc) as price
        from crsp.dsf
        where permno in ({permno_list})
          and date between '{START}' and '{END}'
          and prc is not null
        order by permno, date
    """
    db = wrds.Connection(wrds_username=user, wrds_password=pwd, verbose=False)
    px = db.raw_sql(sql, date_cols=['date']); db.close()
    px.to_csv(PRICE_CACHE, index=False)
    print(f'Saved price cache: {PRICE_CACHE}')
    return px
prices = fetch_crsp_prices_if_needed().dropna(subset=['price'])
prices['permno'] = prices['permno'].astype(int)
prices = prices.sort_values(['permno','date'])
prices['lag_price'] = prices.groupby('permno')['price'].shift(1)
lag_prices = prices[['permno','date','lag_price']].dropna()
print(f'Price rows: {len(prices):,}; lag price rows: {len(lag_prices):,}; firms: {prices.permno.nunique():,}')


Using cached prices: C:\Users\jonat\Lasso_paper\Data\clean_data\crsp_daily_prices_2010_2017_sample.csv
Price rows: 2,608,823; lag price rows: 2,607,527; firms: 1,296


## 3. Construct Quarterly Analyst Revision Fundamental News

In [4]:

def build_fundamental_news_if_needed():
    if FUND_NEWS_CACHE.exists():
        print(f'Using cached fundamental-news panel: {FUND_NEWS_CACHE}')
        return pd.read_csv(FUND_NEWS_CACHE, parse_dates=['date'])
    usecols = ['permno','analys','estimator','fpi','fpedats','anndats','actdats','value','measure','forecast_period_type']
    ibes = pd.read_csv(IBES_PATH, usecols=usecols, parse_dates=['fpedats','anndats','actdats'])
    ibes = ibes[(ibes['measure'].eq('EPS')) & (ibes['forecast_period_type'].eq('Q')) & (ibes['permno'].isin(stocks))].copy()
    ibes = ibes.dropna(subset=['permno','analys','estimator','fpedats','anndats','value'])
    ibes['permno'] = ibes['permno'].astype(int)
    ibes['analyst_id'] = ibes['estimator'].astype('Int64').astype(str) + '_' + ibes['analys'].astype('Int64').astype(str)
    ibes = ibes.sort_values(['permno','analyst_id','fpedats','anndats','actdats'], kind='mergesort')
    ibes['Revision'] = ibes.groupby(['permno','analyst_id','fpedats'])['value'].diff()
    rev = ibes.dropna(subset=['Revision']).copy()
    # Use WRDS activation date when available; it is usually a trading day and aligns to CRSP/predictor dates better than announcement date.
    rev['date'] = rev['actdats'].fillna(rev['anndats'])
    rev = rev.merge(lag_prices, on=['permno','date'], how='left')
    rev = rev[(rev['lag_price'].notna()) & (rev['lag_price'] > 0)].copy()
    rev['ScaledRevision'] = rev['Revision'] / rev['lag_price']
    fund = (rev.groupby(['permno','date'], as_index=False)
              .agg(FundNews=('ScaledRevision','mean'),
                   AbsFundNews=('ScaledRevision', lambda x: np.mean(np.abs(x))),
                   NRevisions=('ScaledRevision','size'),
                   NAnalysts=('analyst_id','nunique')))
    fund.to_csv(FUND_NEWS_CACHE, index=False)
    print(f'Saved fundamental-news panel: {FUND_NEWS_CACHE}')
    print(f'Analyst revision rows used: {len(rev):,}')
    return fund
fund = build_fundamental_news_if_needed(); fund['date'] = pd.to_datetime(fund['date'])
print(f'Firm-day fundamental news rows: {len(fund):,}; firms: {fund.permno.nunique():,}')
print(fund[['FundNews','AbsFundNews','NRevisions']].describe().to_string())


Using cached fundamental-news panel: C:\Users\jonat\Lasso_paper\Data\clean_data\ibes_quarterly_fundamental_news_price_scaled.csv
Firm-day fundamental news rows: 223,463; firms: 979
            FundNews    AbsFundNews     NRevisions
count  223463.000000  223463.000000  223463.000000
mean       -0.011924       0.039202       4.131941
std         0.852939       0.914573       4.499389
min      -147.950880       0.000000       1.000000
25%        -0.001116       0.000424       2.000000
50%        -0.000149       0.001002       3.000000
75%         0.000381       0.002628       4.000000
max       150.793651     150.793651      87.000000


## 4. Construct 5-Day Predictor-Window Exposures

In [5]:

features = pd.read_csv(FEATURE_PATH, parse_dates=['date']).set_index('date').sort_index()
# The HDF5 beta tensor stores predictor names with spaces converted to underscores.
features = features.rename(columns={c: c.replace(' ', '_') for c in features.columns})
missing_predictors = [c for c in topics if c not in features.columns]
if missing_predictors:
    raise ValueError(f'Missing predictors after name normalization: {missing_predictors[:10]} (n={len(missing_predictors)})')
features = features.reindex(columns=topics).astype(float)
exposure = features.shift(1).rolling(WINDOW, min_periods=WINDOW).sum()
exposure.columns = topics
panel_events = fund.merge(exposure.reset_index(), on='date', how='inner').dropna(subset=list(topics))
print(f'Event-exposure rows: {len(panel_events):,}; firms: {panel_events.permno.nunique():,}; predictors: {len(topics):,}')
print(f'Date range: {panel_events.date.min().date()} to {panel_events.date.max().date()}')


Event-exposure rows: 208,947; firms: 979; predictors: 190
Date range: 2010-01-26 to 2017-06-30


## 5. Estimate Firm-Predictor Fundamental Relevance

In [6]:

def compute_fundrel_if_needed():
    if FUNDREL_CACHE.exists():
        print(f'Using cached firm-predictor relevance: {FUNDREL_CACHE}')
        return pd.read_csv(FUNDREL_CACHE)
    rows = []; X_cols = list(topics)
    for permno, g in panel_events.groupby('permno', sort=False):
        n = len(g)
        if n < MIN_EVENTS_PER_FIRM: continue
        y = g['FundNews'].to_numpy(dtype=float); X = g[X_cols].to_numpy(dtype=float)
        y = y - y.mean(); X = X - X.mean(axis=0)
        ssx = np.sum(X * X, axis=0); ok = ssx > 1e-12
        beta = np.full(X.shape[1], np.nan); tval = np.full(X.shape[1], np.nan)
        if ok.any() and n > 2:
            beta[ok] = (X[:, ok].T @ y) / ssx[ok]
            resid = y[:, None] - X[:, ok] * beta[ok]
            sigma2 = np.sum(resid * resid, axis=0) / max(n - 2, 1)
            se = np.sqrt(sigma2 / ssx[ok]); tval[ok] = beta[ok] / se
        rows.append(pd.DataFrame({'permno': permno, 'predictor': X_cols, 'theta': beta, 't_stat': tval, 'FundRel': np.abs(tval), 'n_events': n}))
    fr = pd.concat(rows, ignore_index=True)
    lo, hi = fr['FundRel'].quantile([0.01, 0.99]); fr['FundRel_w'] = fr['FundRel'].clip(lo, hi)
    fr.to_csv(FUNDREL_CACHE, index=False); print(f'Saved firm-predictor relevance: {FUNDREL_CACHE}')
    return fr
fundrel = compute_fundrel_if_needed()
print(f'FundRel rows: {len(fundrel):,}; firms: {fundrel.permno.nunique():,}; predictors: {fundrel.predictor.nunique():,}')
print(fundrel[['FundRel','FundRel_w','n_events']].describe().to_string())


Saved firm-predictor relevance: C:\Users\jonat\Lasso_paper\Data\clean_data\firm_predictor_fundrel_event_window_h5.csv
FundRel rows: 175,370; firms: 923; predictors: 190
             FundRel      FundRel_w       n_events
count  175370.000000  175370.000000  175370.000000
mean        0.970618       0.965198     225.894908
std         0.771825       0.750106     182.823583
min         0.000025       0.014875      20.000000
25%         0.373464       0.373464      98.000000
50%         0.799273       0.799273     176.000000
75%         1.380654       1.380654     291.000000
max         7.723738       3.405974    1069.000000


## 6. Main Firm-Predictor Selection Test

In [7]:

final = selection_df.merge(fundrel, on=['permno','predictor'], how='inner').dropna(subset=['SelectionRate','FundRel_w']).copy()
final.to_csv(FINAL_PANEL_PATH, index=False)
print(f'Final firm-predictor panel rows: {len(final):,}; firms: {final.permno.nunique():,}; predictors: {final.predictor.nunique():,}')
def twoway_demean(df, col, fe1='permno', fe2='predictor'):
    return df[col] - df.groupby(fe1)[col].transform('mean') - df.groupby(fe2)[col].transform('mean') + df[col].mean()
final['y_dm'] = twoway_demean(final, 'SelectionRate')
final['x_dm'] = twoway_demean(final, 'FundRel_w')
main_model = sm.OLS(final['y_dm'], final[['x_dm']]).fit(cov_type='cluster', cov_kwds={'groups': final['permno']})
beta = main_model.params['x_dm']; se = main_model.bse['x_dm']; t = main_model.tvalues['x_dm']; p = main_model.pvalues['x_dm']
print('Main test: SelectionRate_{i,j} on FundRel_{i,j} with firm and predictor FEs')
print(f'beta = {beta:.6g}, se(cluster firm) = {se:.6g}, t = {t:.3f}, p = {p:.4g}')
print(f'Interpretation: one unit higher |t|-based FundRel is associated with {beta*100:.4f} percentage points higher selection rate.')


Final firm-predictor panel rows: 175,370; firms: 923; predictors: 190
Main test: SelectionRate_{i,j} on FundRel_{i,j} with firm and predictor FEs
beta = 0.000169672, se(cluster firm) = 0.00011818, t = 1.436, p = 0.1511
Interpretation: one unit higher |t|-based FundRel is associated with 0.0170 percentage points higher selection rate.


## 7. Predictor-Level Validation

In [8]:

events = panel_events.copy(); events['month'] = events['date'].dt.to_period('M').astype(str)
events['FundNews_resid'] = events['FundNews'] - events.groupby('permno')['FundNews'].transform('mean') - events.groupby('month')['FundNews'].transform('mean') + events['FundNews'].mean()
pred_rows = []
for pred in topics:
    x = events[pred]
    x_resid = x - events.groupby('permno')[pred].transform('mean') - events.groupby('month')[pred].transform('mean') + x.mean()
    tmp = pd.DataFrame({'y': events['FundNews_resid'], 'x': x_resid, 'date': events['date']}).dropna()
    if tmp['x'].var() <= 1e-14:
        pred_rows.append((pred, np.nan, np.nan, np.nan)); continue
    m = sm.OLS(tmp['y'], sm.add_constant(tmp['x'])).fit(cov_type='cluster', cov_kwds={'groups': tmp['date']})
    pred_rows.append((pred, m.params['x'], m.tvalues['x'], m.pvalues['x']))
predrel = pd.DataFrame(pred_rows, columns=['predictor','theta_j','t_j','p_j']); predrel['FundRel_j'] = predrel['t_j'].abs()
predsel = final.groupby('predictor', as_index=False)['SelectionRate'].mean().rename(columns={'SelectionRate':'SelectionRate_j'})
predval = predsel.merge(predrel, on='predictor').dropna()
pred_model = sm.OLS(predval['SelectionRate_j'], sm.add_constant(predval['FundRel_j'])).fit(cov_type='HC1')
print(f'Predictor-level validation N={len(predval)}')
print(f'b = {pred_model.params["FundRel_j"]:.6g}, se(HC1) = {pred_model.bse["FundRel_j"]:.6g}, t = {pred_model.tvalues["FundRel_j"]:.3f}, p = {pred_model.pvalues["FundRel_j"]:.4g}')
print('Top 10 aggregate FundRel predictors:')
print(predval.sort_values('FundRel_j', ascending=False).head(10)[['predictor','FundRel_j','SelectionRate_j']].to_string(index=False))


Predictor-level validation N=190
b = 0.00134457, se(HC1) = 0.00155745, t = 0.863, p = 0.388
Top 10 aggregate FundRel predictors:
           predictor  FundRel_j  SelectionRate_j
             Buffett   3.439723         0.017211
       Cultural_life   3.235005         0.019647
        Credit_cards   2.586376         0.018994
       Announce_plan   2.520263         0.074866
   Scenario_analysis   2.348168         0.018108
            Key_role   2.312434         0.021538
    Trade_agreements   2.296389         0.015319
            Pensions   2.286507         0.025261
VIX_Volatility_Index   2.273640         0.079862
           Watchdogs   2.213001         0.021490


## 8. Save Summary

In [9]:

summary = {
    'window_trading_days': WINDOW, 'min_events_per_firm': MIN_EVENTS_PER_FIRM,
    'selection_rows': int(len(selection_df)), 'fundamental_news_rows': int(len(fund)),
    'event_exposure_rows': int(len(panel_events)), 'fundrel_rows': int(len(fundrel)),
    'final_rows': int(len(final)), 'final_firms': int(final.permno.nunique()), 'final_predictors': int(final.predictor.nunique()),
    'main_beta': float(beta), 'main_se_cluster_firm': float(se), 'main_t': float(t), 'main_p': float(p),
    'predictor_level_beta': float(pred_model.params['FundRel_j']), 'predictor_level_se_hc1': float(pred_model.bse['FundRel_j']),
    'predictor_level_t': float(pred_model.tvalues['FundRel_j']), 'predictor_level_p': float(pred_model.pvalues['FundRel_j']),
    'final_panel_path': str(FINAL_PANEL_PATH), 'fundamental_news_path': str(FUND_NEWS_CACHE), 'fundrel_path': str(FUNDREL_CACHE),
}
SUMMARY_PATH.write_text(json.dumps(summary, indent=2))
print(json.dumps(summary, indent=2))


{
  "window_trading_days": 5,
  "min_events_per_firm": 20,
  "selection_rows": 246240,
  "fundamental_news_rows": 223463,
  "event_exposure_rows": 208947,
  "fundrel_rows": 175370,
  "final_rows": 175370,
  "final_firms": 923,
  "final_predictors": 190,
  "main_beta": 0.0001696720655571013,
  "main_se_cluster_firm": 0.00011817984811948314,
  "main_t": 1.4357106415093552,
  "main_p": 0.15108469590159215,
  "predictor_level_beta": 0.001344565521357389,
  "predictor_level_se_hc1": 0.001557447801039247,
  "predictor_level_t": 0.8633133774757606,
  "predictor_level_p": 0.3879651910298221,
  "final_panel_path": "C:\\Users\\jonat\\Lasso_paper\\Data\\clean_data\\correlated_predictors_firm_predictor_panel.csv",
  "fundamental_news_path": "C:\\Users\\jonat\\Lasso_paper\\Data\\clean_data\\ibes_quarterly_fundamental_news_price_scaled.csv",
  "fundrel_path": "C:\\Users\\jonat\\Lasso_paper\\Data\\clean_data\\firm_predictor_fundrel_event_window_h5.csv"
}
